# Understanding Partial Least Squares (PLS)

In the [SVD notebook](../part0_intro_to_svd/intro_to_svd.ipynb) we met the Singular Value Decomposition: take a _single_ matrix and break it into orthogonal directions, ranked by how much _variance_ each one explains. Keep the top few and you have a compact, low-rank summary of the matrix. That single idea is the engine behind an astonishing amount of the PEST/PEST++ world.

This notebook is about a close cousin of SVD: **Partial Least Squares (PLS)**. Where SVD looks at _one_ matrix and finds directions of maximum _variance_, PLS looks at _two_ matrices and finds directions of maximum _covariance_ - the directions along which the two matrices move _together_.

Why do we care in groundwater modelling? Because we are constantly staring at two matrices that are lined up realization-by-realization:

- $\mathbf{X}$ - a **parameter ensemble**: one row per realization, one column per adjustable parameter (hydraulic conductivity pilot points, recharge multipliers, storage, ...).
- $\mathbf{Y}$ - a **prediction/forecast/output ensemble**: the same rows (realizations), but now the columns are the model outputs we care about (a stream depletion, a head at a well, a particle travel time, ...).

PLS finds the low-dimensional structure that _connects_ the parameters to the predictions, and packages it into a **prediction matrix** $\mathbf{B}$ so that

$$\mathbf{\hat{Y}} = \mathbf{X}\,\mathbf{B}$$

is a cheap, linear **emulator** - no model run required. This is what `pyemu`'s PLS emulator does under the hood (see the applied [PLS emulator notebook](../part2_10_eva_and_dsi/3_freyberg_ensemble_pls_emulator.ipynb)); here we build it from scratch so you can see every gear turning.

Along the way we will keep pointing back at SVD, because PLS _is_ SVD in disguise: the very first thing PLS computes turns out to be a singular vector of the cross-covariance between $\mathbf{X}$ and $\mathbf{Y}$.

## Getting ready

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
np.random.seed(7)

# PLS is built on the NIPALS algorithm; sklearn ships a battle-tested version we will
# check our from-scratch code against. pyemu.emulators.PLS wraps this same sklearn class.
from sklearn.cross_decomposition import PLSRegression

## Step 1: cook up two ensemble matrices

Let's make a synthetic - but groundwater-flavoured - pair of ensembles so we know the "true" structure and can watch PLS recover it.

The trick to building realistic ensembles is that both the parameters and the predictions are, deep down, driven by a _small number of hidden "drivers"_. Think of it this way: the historical data and the geology don't really let every parameter wander independently - a handful of underlying controls (regional transmissivity, the recharge regime, the storage regime) set the tone, and the individual parameters and forecasts inherit their behaviour from those controls.

We will use **3 hidden drivers** `F` (these are _latent_ - PLS never sees them), and build:

- $\mathbf{X}$: 10 parameters - 4 K pilot points, 3 recharge zones, 3 storage terms - each loading mostly on one driver (this creates **intra**-correlation _within_ the parameters), plus a bit of noise.
- $\mathbf{Y}$: 4 forecasts, each a different mixture of the same 3 drivers (this creates **inter**-correlation _between_ parameters and forecasts, and **intra**-correlation _within_ the forecasts).

In [ ]:
n = 500          # number of realizations
k_true = 3       # number of hidden drivers (PLS does not know this)

par_names  = ["k_pp1","k_pp2","k_pp3","k_pp4","rch_z1","rch_z2","rch_z3","ss_1","ss_2","ss_3"]
pred_names = ["sw_depletion","head_well_A","head_well_B","travel_time"]
p, q = len(par_names), len(pred_names)

# the hidden drivers: one value per realization per driver
F = np.random.normal(size=(n, k_true))

# parameter loadings: each block of params loads mostly on ONE driver (block structure)
Wx = np.zeros((k_true, p))
Wx[0, 0:4]  = np.random.uniform(0.7, 1.3, 4)   # K pilot points  <- driver 0 ("regional K")
Wx[1, 4:7]  = np.random.uniform(0.7, 1.3, 3)   # recharge zones  <- driver 1 ("recharge")
Wx[2, 7:10] = np.random.uniform(0.7, 1.3, 3)   # storage terms   <- driver 2 ("storage")
Wx += 0.15 * np.random.normal(size=(k_true, p))  # a little cross-loading

X = pd.DataFrame(F @ Wx + 0.25*np.random.normal(size=(n, p)), columns=par_names)

# prediction loadings: each forecast is its own mixture of the 3 drivers
Wy = np.random.normal(size=(k_true, q))
Y = pd.DataFrame(F @ Wy + 0.25*np.random.normal(size=(n, q)), columns=pred_names)

print("parameter ensemble X:", X.shape, " (realizations x parameters)")
print("prediction ensemble Y:", Y.shape, " (realizations x forecasts)")
X.head()

### Look at the correlation structure

Because everything descends from 3 hidden drivers, the parameters and predictions are richly correlated. Let's stack $\mathbf{X}$ and $\mathbf{Y}$ side by side and look at the correlation matrix. This is the structure PLS is going to exploit.

In [ ]:
both = pd.concat([X, Y], axis=1)
corr = both.corr()

fig, ax = plt.subplots(1, 1, figsize=(8, 7))
im = ax.imshow(corr, cmap="coolwarm", vmin=-1, vmax=1)
ax.set_xticks(range(corr.shape[1])); ax.set_xticklabels(corr.columns, rotation=90)
ax.set_yticks(range(corr.shape[0])); ax.set_yticklabels(corr.columns)
# draw a box separating parameters (X) from predictions (Y)
ax.axhline(p-0.5, color="k", lw=2); ax.axvline(p-0.5, color="k", lw=2)
ax.set_title("correlation of [ X | Y ]\ntop-left block = intra-X, bottom-right = intra-Y,\noff-diagonal blocks = inter (X<->Y)")
plt.colorbar(im, ax=ax, shrink=0.7)
plt.tight_layout(); plt.show()

Three things to notice, and they map exactly onto the three kinds of relationship the notebook is about:

- **intra-X** (top-left block): the K pilot points are correlated with each other, the recharge zones with each other, etc. - the block structure from the drivers.
- **intra-Y** (bottom-right block): the forecasts are correlated with each other too, because they share drivers.
- **inter X<->Y** (the off-diagonal blocks): parameters and forecasts co-vary. _This_ is the signal PLS lives on - it is what lets parameters predict forecasts.

## Step 2: centre and scale

Just like SVD, PLS works on **mean-centred** data - we care about how realizations vary _around_ the ensemble mean, not the mean itself. (This should feel familiar: it is exactly the ensemble _anomaly_ matrix that shows up all over `PESTPP-IES`.) We also **scale** each column to unit standard deviation so that a parameter measured in "big numbers" doesn't dominate one measured in "small numbers" - PLS should care about _correlation_, not units.

In [ ]:
def standardize(A, mu=None, sd=None):
    """centre and scale to unit std; return the scaled array plus the mean/std to undo it later"""
    A = np.asarray(A, dtype=float)
    if mu is None:
        mu = A.mean(axis=0); sd = A.std(axis=0, ddof=1)
    return (A - mu) / sd, mu, sd

Xc, xmu, xsd = standardize(X)
Yc, ymu, ysd = standardize(Y)
print("centred & scaled: X mean ~", np.round(Xc.mean(0),3)[:3], " std ~", np.round(Xc.std(0),3)[:3])

## Step 3: the key idea (and the SVD connection)

PLS looks for a direction $\mathbf{w}$ in _parameter_ space and a direction $\mathbf{c}$ in _prediction_ space such that the **scores**

$$\mathbf{t} = \mathbf{X}\mathbf{w} \qquad \text{(a single number per realization: where it sits along } \mathbf{w}\text{)}$$
$$\mathbf{u} = \mathbf{Y}\mathbf{c}$$

**co-vary as strongly as possible**. In other words, PLS maximizes $\text{cov}(\mathbf{t}, \mathbf{u}) = \mathbf{w}^T (\mathbf{X}^T\mathbf{Y}) \mathbf{c}$.

Here is the punchline, and it is pure SVD:

> The pair $(\mathbf{w}, \mathbf{c})$ that maximizes $\mathbf{w}^T (\mathbf{X}^T\mathbf{Y}) \mathbf{c}$ is exactly the **leading left and right singular vectors of the cross-covariance matrix $\mathbf{X}^T\mathbf{Y}$**.

Remember from the SVD notebook: the top singular vectors of a matrix point along its dominant direction. PLS just applies that to the _cross_-covariance $\mathbf{X}^T\mathbf{Y}$ instead of a single matrix's variance. Let's see it directly.

In [ ]:
# cross-covariance between parameters and predictions
Cxy = Xc.T @ Yc            # shape (p, q)

# SVD of the cross-covariance - straight out of the SVD notebook
U_svd, s_svd, Vt_svd = np.linalg.svd(Cxy)
w_svd = U_svd[:, 0]        # leading left singular vector  -> parameter direction
c_svd = Vt_svd[0, :]       # leading right singular vector -> prediction direction

print("leading singular vector of X^T Y (the PLS parameter direction w):")
print(pd.Series(np.round(w_svd, 2), index=par_names))

Notice how that first direction loads on the K pilot points and (a bit) the others - it is the single parameter-space direction most predictive of the forecasts. We could stop here, but there is a beautiful iterative algorithm - **NIPALS** - that finds this same direction _and_ sets us up to peel off the next one, and the next, one component at a time. Let's build it.

## Step 4: NIPALS, one component at a time

NIPALS (Nonlinear Iterative PArtial Least Squares) finds that leading direction _without_ ever forming a full SVD, using a simple **power-iteration** loop. For a single component:

1. start with a guess for the prediction score $\mathbf{u}$ (just use a column of $\mathbf{Y}$),
2. $\mathbf{w} = \mathbf{X}^T\mathbf{u}$, then normalize -- the parameter direction that best matches $\mathbf{u}$,
3. $\mathbf{t} = \mathbf{X}\mathbf{w}$ -- the parameter scores,
4. $\mathbf{c} = \mathbf{Y}^T\mathbf{t}\,/\,(\mathbf{t}^T\mathbf{t})$ -- the prediction direction that best matches $\mathbf{t}$,
5. $\mathbf{u} = \mathbf{Y}\mathbf{c}\,/\,(\mathbf{c}^T\mathbf{c})$ -- updated prediction scores,
6. repeat 2-5 until $\mathbf{t}$ stops changing.

This "bounce back and forth between X and Y until it settles" loop is exactly power iteration converging on the top singular triplet of $\mathbf{X}^T\mathbf{Y}$ - the same $\mathbf{w}$ we just got from the SVD. (Like all power iteration, how fast it settles depends on how dominant the leading direction is - it can take anywhere from a few to a few dozen sweeps.) Let's watch it converge.

In [ ]:
# --- NIPALS inner loop for the FIRST component, printing convergence ---
Xw, Yw = Xc.copy(), Yc.copy()     # working copies we will "deflate" later

u = Yw[:, 0].copy()               # start: first prediction column
t_prev = np.zeros(Xw.shape[0])
for it in range(200):
    w = Xw.T @ u; w /= np.linalg.norm(w)     # (2) parameter direction, normalized
    t = Xw @ w                                # (3) parameter scores
    c = Yw.T @ t / (t @ t)                    # (4) prediction direction
    u = Yw @ c / (c @ c)                      # (5) prediction scores
    change = np.linalg.norm(t - t_prev)
    if it < 4 or it % 15 == 0 or change < 1e-10:
        print("iter {0:>3d}: change in t = {1:.2e}".format(it, change))
    if change < 1e-10:
        break
    t_prev = t.copy()

# align sign with the SVD result and compare
if np.dot(w, w_svd) < 0:
    w = -w; t = -t; c = -c; u = -u
print("\nNIPALS w matches SVD(X^T Y) w?  max abs diff = {0:.2e}".format(np.max(np.abs(w - w_svd))))

It settles down (the change in $\mathbf{t}$ marching towards zero), and the direction it lands on is - to machine precision - the leading singular vector of $\mathbf{X}^T\mathbf{Y}$. **NIPALS is power iteration on the cross-covariance.** That is the SVD linkage made concrete.

### Deflation: peeling off a component

Once we have the first component's scores $\mathbf{t}$, we _remove_ what it explains from both $\mathbf{X}$ and $\mathbf{Y}$ (this is called **deflation**), and run the same loop again on the leftovers to get the second component, and so on. This is the direct analogue of SVD peeling off one singular value/vector at a time - each PLS component is orthogonal to the last.

To deflate we need the **X-loading** $\mathbf{p} = \mathbf{X}^T\mathbf{t}/(\mathbf{t}^T\mathbf{t})$ (how the parameters project back onto the score), then subtract:

In [ ]:
p_load = Xw.T @ t / (t @ t)      # X-loading for component 1
Xw = Xw - np.outer(t, p_load)    # remove component 1 from X
Yw = Yw - np.outer(t, c)         # remove component 1 from Y
print("variance remaining in X after deflating 1 component: {0:.1%}".format(
      (Xw**2).sum() / (Xc**2).sum()))
print("variance remaining in Y after deflating 1 component: {0:.1%}".format(
      (Yw**2).sum() / (Yc**2).sum()))

### Wrap the whole thing in a function

Putting the inner loop and the deflation together, here is the complete NIPALS PLS - it extracts `n_comp` components and hands back the four matrices we need:

- $\mathbf{W}$ - the parameter directions (X-weights), one column per component,
- $\mathbf{T}$ - the parameter scores,
- $\mathbf{P}$ - the parameter loadings (for deflation / reconstruction),
- $\mathbf{C}$ - the prediction directions/loadings.

In [ ]:
def nipals_pls(X, Y, n_comp, tol=1e-11, itmax=500):
    """from-scratch NIPALS PLS on centred+scaled X, Y. Returns W, T, P, C."""
    X = X.copy(); Y = Y.copy()
    n, p = X.shape; q = Y.shape[1]
    W = np.zeros((p, n_comp)); T = np.zeros((n, n_comp))
    P = np.zeros((p, n_comp)); C = np.zeros((q, n_comp))
    for a in range(n_comp):
        u = Y[:, 0].copy()
        for _ in range(itmax):
            w = X.T @ u; w /= np.linalg.norm(w)   # parameter direction
            t = X @ w                              # parameter scores
            c = Y.T @ t / (t @ t)                  # prediction direction
            u_new = Y @ c / (c @ c)                # prediction scores
            if np.linalg.norm(u_new - u) < tol:
                u = u_new; break
            u = u_new
        p_load = X.T @ t / (t @ t)                 # parameter loading
        X = X - np.outer(t, p_load)                # deflate X
        Y = Y - np.outer(t, c)                     # deflate Y
        W[:, a] = w; T[:, a] = t; P[:, a] = p_load; C[:, a] = c
    return W, T, P, C

W, T, P, C = nipals_pls(Xc, Yc, n_comp=3)
print("W (parameter directions):", W.shape, " T (scores):", T.shape, " C (prediction directions):", C.shape)

### What did the components find?

Because we cooked up the data with 3 block-structured drivers, we expect the parameter directions ($\mathbf{W}$) to line up with those blocks: one component dominated by the K pilot points, one by recharge, one by storage. Let's plot the weights.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 3.2), sharey=True)
for a in range(3):
    axes[a].bar(range(p), W[:, a], color=["C0"]*4 + ["C1"]*3 + ["C2"]*3)
    axes[a].set_xticks(range(p)); axes[a].set_xticklabels(par_names, rotation=90)
    axes[a].axhline(0, color="k", lw=0.6)
    axes[a].set_title("component {0}\n(parameter direction w{0})".format(a+1))
axes[0].set_ylabel("weight")
fig.suptitle("PLS parameter directions - each picks out a group of related parameters", y=1.05)
plt.tight_layout(); plt.show()

Each component isolates a coherent _group_ of parameters - the latent drivers, recovered. This is the PLS analogue of an SVD "mode": a single interpretable direction that summarises many correlated columns.

The plot of scores makes the low-dimensionality vivid: every realization, originally described by 10 parameter values, is now well-summarised by just 3 score numbers.

## Step 5: forming the prediction matrix

We now have everything we need to build the emulator. In the PLS latent space, the relationship between parameters and predictions is captured by combining the weights, loadings, and prediction directions into a single **regression (prediction) matrix**:

$$\mathbf{B} = \mathbf{W}\,(\mathbf{P}^T\mathbf{W})^{-1}\,\mathbf{C}^T$$

The clever bit is _which_ inverse we take. $\mathbf{P}^T\mathbf{W}$ is a tiny `n_comp`-by-`n_comp` matrix (here just 3x3), so it is trivially and _stably_ invertible - no matter how many parameters we have. Compare that with ordinary least squares, which would need to invert the full $\mathbf{X}^T\mathbf{X}$ (a `p`-by-`p` matrix). We will see in a moment why that matters enormously in groundwater problems.

Once we have $\mathbf{B}$ (defined on the centred+scaled data), predicting is just a matrix multiply, followed by undoing the scaling:

In [ ]:
def make_prediction_matrix(W, P, C):
    """the PLS regression matrix B such that (scaled) Yhat = (scaled) X @ B"""
    return W @ np.linalg.inv(P.T @ W) @ C.T

def pls_predict(Xnew, W, P, C, xmu, xsd, ymu, ysd):
    """emulate forecasts for new (unscaled) parameter sets"""
    B = make_prediction_matrix(W, P, C)
    Xnew_c = (np.asarray(Xnew, dtype=float) - xmu) / xsd   # scale inputs
    Yhat_c = Xnew_c @ B                                    # predict in scaled space
    return Yhat_c * ysd + ymu                              # un-scale to real units

B = make_prediction_matrix(W, P, C)
print("prediction matrix B:", B.shape, " (parameters x forecasts)")

The prediction matrix is `p`-by-`q` (10 parameters by 4 forecasts). Each column tells you how a forecast responds to each parameter - a compact, linear, run-free stand-in for the model. Let's sanity-check it against `sklearn` (and hence against `pyemu`'s emulator, which wraps the same code).

In [ ]:
pls_sk = PLSRegression(n_components=3, scale=True)
pls_sk.fit(X.values, Y.values)

Yhat_ours = pls_predict(X.values, W, P, C, xmu, xsd, ymu, ysd)
Yhat_sk   = pls_sk.predict(X.values)
print("our from-scratch PLS vs sklearn PLSRegression: max abs difference in predictions = {0:.2e}".format(
      np.max(np.abs(Yhat_ours - Yhat_sk))))

Identical to ~1e-6. Our from-scratch NIPALS _is_ the library.

## Step 6: using the prediction matrix as an emulator

The whole point of $\mathbf{B}$ is prediction on **new** parameter sets we have never run through the model. Let's test that honestly: train PLS on part of the ensemble, then emulate the forecasts for held-out realizations and compare against their true values.

In [ ]:
# split into training and testing realizations
ntr = 400
Xtr, Xte = X.values[:ntr], X.values[ntr:]
Ytr, Yte = Y.values[:ntr], Y.values[ntr:]

# fit PLS on the TRAINING ensemble only
Xc_tr, xmu, xsd = standardize(Xtr)
Yc_tr, ymu, ysd = standardize(Ytr)
W, T, P, C = nipals_pls(Xc_tr, Yc_tr, n_comp=3)

# emulate the held-out (unseen) realizations
Yte_hat = pls_predict(Xte, W, P, C, xmu, xsd, ymu, ysd)

fig, axes = plt.subplots(1, q, figsize=(13, 3.2))
for j in range(q):
    axes[j].scatter(Yte[:, j], Yte_hat[:, j], s=12, alpha=0.6)
    lo, hi = Yte[:, j].min(), Yte[:, j].max()
    axes[j].plot([lo, hi], [lo, hi], "k--", lw=1)
    r2 = 1 - np.sum((Yte[:, j]-Yte_hat[:, j])**2)/np.sum((Yte[:, j]-Yte[:, j].mean())**2)
    axes[j].set_title("{0}\ntest R2 = {1:.2f}".format(pred_names[j], r2))
    axes[j].set_xlabel("true");
axes[0].set_ylabel("emulated")
fig.suptitle("PLS emulator on held-out realizations (3 components)", y=1.05)
plt.tight_layout(); plt.show()

The emulator predicts forecasts for realizations it never saw - a run-free stand-in for the model. That is the same job `pyemu`'s PLS emulator does on a real Freyberg ensemble.

### How many components? (the SVD truncation curve)

In the SVD notebook we chose _how many singular values to keep_ by watching reconstruction error fall off and then plateau. PLS has the exact same knob: the **number of components**. Too few and we miss real signal; too many and we start fitting noise. Let's sweep it and watch train vs test accuracy.

In [ ]:
ks = range(1, p+1)
train_r2, test_r2 = [], []
def r2(Yt, Yh): return 1 - np.sum((Yt-Yh)**2)/np.sum((Yt-Yt.mean(0))**2)
for kc in ks:
    Wk, Tk, Pk, Ck = nipals_pls(Xc_tr, Yc_tr, n_comp=kc)
    train_r2.append(r2(Ytr, pls_predict(Xtr, Wk, Pk, Ck, xmu, xsd, ymu, ysd)))
    test_r2.append( r2(Yte, pls_predict(Xte, Wk, Pk, Ck, xmu, xsd, ymu, ysd)))

fig, ax = plt.subplots(1, 1, figsize=(6, 4))
ax.plot(list(ks), train_r2, "o-", label="training R2")
ax.plot(list(ks), test_r2,  "s-", label="test R2 (held-out)")
ax.axvline(k_true, color="r", ls="--", label="true # drivers = {0}".format(k_true))
ax.set_xlabel("number of PLS components"); ax.set_ylabel("R2")
ax.set_title("accuracy vs number of components\n(cf. keeping singular values in SVD)")
ax.legend(); ax.grid()
plt.tight_layout(); plt.show()

The accuracy climbs steeply, then flattens right around **3 components** - exactly the number of hidden drivers we built in. Beyond that we are just adding components that chase noise (train R2 keeps inching up while test R2 stops improving). This is the same "keep the important modes, drop the rest" trade-off as SVD truncation - PLS just ranks its modes by _covariance with the forecasts_ instead of by raw variance.

## Step 7: why PLS instead of plain least squares?

You might ask: why not just do ordinary least squares - fit $\mathbf{Y} = \mathbf{X}\mathbf{B}$ directly by inverting $\mathbf{X}^T\mathbf{X}$? In our toy problem (10 parameters, 400 realizations) you could. But real groundwater parameter ensembles are the opposite shape: **thousands of parameters, hundreds of realizations** ($p \gg n$). Here is what that does to $\mathbf{X}^T\mathbf{X}$.

In [ ]:
# high-dimensional, groundwater-like regime: many more parameters than realizations
p_big, n_big = 300, 80
Fb = np.random.normal(size=(n_big, 3))
Xbig = Fb @ np.random.normal(size=(3, p_big)) + 0.3*np.random.normal(size=(n_big, p_big))

XtX = Xbig.T @ Xbig                      # the matrix OLS must invert  (p_big x p_big)
sv = np.linalg.svd(XtX, compute_uv=False)  # <- straight from the SVD notebook
n_zero = int((sv < 1e-8*sv.max()).sum())
print("X is {0} realizations x {1} parameters".format(n_big, p_big))
print("X^T X is {0}x{0}, but its rank is at most {1}".format(p_big, n_big))
print("  -> {0} of its {1} singular values are ~zero".format(n_zero, p_big))
print("  -> condition number ~ {0:.1e}  (effectively singular - OLS cannot invert it)".format(
      sv.max()/sv[sv>1e-12].min()))

The SVD of $\mathbf{X}^T\mathbf{X}$ tells the whole story (just like it did for images in the SVD notebook): with more parameters than realizations, most of its singular values are essentially zero, so it is **singular** - ordinary least squares simply cannot form its inverse, and any regularized version is exquisitely sensitive to noise.

PLS sidesteps this entirely. It never inverts $\mathbf{X}^T\mathbf{X}$; it only inverts the tiny `n_comp`-by-`n_comp` matrix $\mathbf{P}^T\mathbf{W}$. By working in a handful of covariance-ranked latent directions - the SVD-flavoured modes of $\mathbf{X}^T\mathbf{Y}$ - it stays well-posed and noise-robust no matter how many parameters you throw at it. That is precisely why PLS is a workhorse emulator for high-dimensional groundwater problems.

## Recap

- **PLS is SVD's two-matrix cousin.** SVD finds directions of maximum _variance_ in one matrix; PLS finds directions of maximum _covariance_ between a parameter ensemble $\mathbf{X}$ and a prediction ensemble $\mathbf{Y}$.
- We built two ensembles with **intra**-correlation (within X, within Y) and **inter**-correlation (X<->Y), all descending from a few hidden drivers - the situation we always face in groundwater modelling.
- **NIPALS** extracts the latent components one at a time by simple power iteration; its first parameter direction is exactly the **leading singular vector of the cross-covariance $\mathbf{X}^T\mathbf{Y}$** - the SVD link made concrete. Deflation peels off components just like SVD peels off singular triplets.
- The components assemble into a **prediction matrix** $\mathbf{B}$: a cheap, run-free, linear emulator mapping parameters straight to forecasts, matching `sklearn`/`pyemu` to machine precision.
- Choosing the **number of components** is the same accuracy-vs-parsimony trade-off as **SVD truncation** - keep the modes that matter, drop the ones that chase noise.
- Because PLS only ever inverts a tiny latent-space matrix, it thrives in the **$p \gg n$** regime where ordinary least squares (which must invert the singular $\mathbf{X}^T\mathbf{X}$) falls apart.

### Where to next

- [Intro to SVD](../part0_intro_to_svd/intro_to_svd.ipynb) - the single-matrix foundation everything here builds on.
- [The applied PLS emulator notebook](../part2_10_eva_and_dsi/3_freyberg_ensemble_pls_emulator.ipynb) - `pyemu.emulators.PLS` on a real Freyberg ensemble, cross-validated and wired into a `PEST++` forward run.
- [Data Space Inversion](../part2_10_eva_and_dsi/2_freyberg_ensemble_data_space_inversion.ipynb) - a different emulator that works purely on the _outputs_.